In [13]:
from pathlib import Path
import geopandas as gpd
import pandas as pd


caminho = Path("../dados/brutos/prefeitura/sedes_das_prefeituras_bairro_v1_p.gpkg")

print(Path.exists(caminho))
camadas = gpd.list_layers(caminho)

print(camadas.to_string(index=False))

for camada in camadas.itertuples(index=False):
    if pd.isna(camada.geometry_type):
        continue

    dados = gpd.read_file(caminho, layer=camada.name)
    print("\nCamada:", camada.name)
    print("Registros:", len(dados))
    print("CRS:", dados.crs)
    print("Colunas:", dados.columns.tolist())
    print(dados.drop(columns=dados.geometry.name).head(3).to_string(index=False))

True
                             name geometry_type
sedes_das_prefeituras_bairro_v1_p         Point

Camada: sedes_das_prefeituras_bairro_v1_p
Registros: 13
CRS: EPSG:31984
Colunas: ['cod_orgpub', 'sigla', 'nome', 'logradouro', 'num_porta', 'complement', 'bairro', 'pref_bairr', 'cep', 'codlog', 'cod_bairro', 'street_vie', 'servicos', 'x', 'y', 'servicos_pref', 'bairros_abrangidos', 'telefone', 'email', 'diretor', 'geometry']
 cod_orgpub                sigla                                                       nome           logradouro num_porta                                complement           bairro                              pref_bairr       cep  codlog  cod_bairro                                                                                                                                                                                          street_vie                                                                                                                           

In [14]:
sedes = gpd.read_file(
    caminho,
    layer="sedes_das_prefeituras_bairro_v1_p",
)

print(
    sedes[
        ["cod_orgpub", "sigla", "nome", "pref_bairr"]
    ].to_string(index=False)
)

print("\nGeometrias ausentes:", sedes.geometry.isna().sum())
print("Geometrias vazias:", sedes.geometry.is_empty.sum())
print("Geometrias inválidas:", (~sedes.geometry.is_valid).sum())

 cod_orgpub                  sigla                                                       nome                                       pref_bairr
       11.0                  SACPB Secretaria de Articulação Comunitária e Prefeituras-Bairro            Prefeitura-Bairro I - Centro / Brotas
        1.0    PREFEITURA BAIRRO I                        Prefeitura Bairro I - CENTRO/BROTAS            Prefeitura-Bairro I - Centro / Brotas
        2.0   PREFEITURA BAIRRO II                      Prefeitura Bairro II - SUBÚRBIO/ILHAS          Prefeitura-Bairro II - Subúrbio / Ilhas
        4.0   PREFEITURA BAIRRO IV                     Prefeitura Bairro IV - ITAPUÃ/IPITANGA         Prefeitura-Bairro IV - Itapuã / Ipitanga
        5.0    PREFEITURA BAIRRO V                         Prefeitura Bairro V - CIDADE BAIXA               Prefeitura-Bairro V - Cidade Baixa
        3.0  PREFEITURA BAIRRO III                         Prefeitura Bairro III - CAJAZEIRAS               Prefeitura-Bairro III - Cajazeiras

In [15]:
siglas = sedes["sigla"].astype("string").str.strip().str.upper()

sedes_regionais = sedes.loc[
    siglas.str.fullmatch(r"PREFEITURA BAIRRO [IVX]+", na=False)
].copy()

sedes_regionais["cod_pbr"] = (
    sedes_regionais["sigla"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.extract(r"PREFEITURA BAIRRO ([IVX]+)$", expand=False)
)

codigos_esperados = {"I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"}

if (
    len(sedes_regionais) != 10
    or sedes_regionais["cod_pbr"].duplicated().any()
    or set(sedes_regionais["cod_pbr"]) != codigos_esperados
):
    raise ValueError("Confira as dez sedes e seus códigos regionais.")

print("Sedes selecionadas:", len(sedes_regionais))
print(
    sedes_regionais[
        ["cod_pbr", "nome", "logradouro", "num_porta", "complement", "bairro"]
    ].to_string(index=False)
)

Sedes selecionadas: 10
cod_pbr                                           nome                              logradouro num_porta                                             complement           bairro
      I            Prefeitura Bairro I - CENTRO/BROTAS                    Rua Guedes de Britto         1                                   Ed. Ranulfo Oliveira Centro Histórico
     II          Prefeitura Bairro II - SUBÚRBIO/ILHAS                                Rua Pará        15                    Ao lado esquerdo da Igreja Luterana           Paripe
     IV         Prefeitura Bairro IV - ITAPUÃ/IPITANGA                  Avenida Dorival Caymmi        17                                        Proximo ao INSS           Itapuã
      V             Prefeitura Bairro V - CIDADE BAIXA               Avenida Porto dos Mastros        65         Em frente a Biblioteca Municipal Edgard Santos          Ribeira
    III             Prefeitura Bairro III - CAJAZEIRAS Avenida Engenheiro Raymundo Carlos Ne